# Final evaluation — chỉ chạy sau khi chốt thí nghiệm
Chốt model/config/seeds và, với raw protocol, bidder policy trước khi mở test.
Đổi lựa chọn model sau khi xem kết quả test sẽ phá vỡ protocol.
Notebook mặc định khóa cell final bằng `RUN_FINAL_TEST = False`. Đổi thành True khi nghiên cứu đã chốt.
Với chronological data, điều chỉnh DATA_DIR và CACHE tương ứng trước khi khôi phục.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile

"""Standard-library helpers embedded in Colab cells before source is extracted."""

import hashlib
from pathlib import Path
import tempfile
import zipfile
import json


def file_sha256(path):
    checksum = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            checksum.update(chunk)
    return checksum.hexdigest()


def restore_project(archive_path, destination):
    """Publish a complete extraction, or reject stale/modified local source."""
    archive_path, destination = Path(archive_path), Path(destination)
    archive_hash = file_sha256(archive_path)
    receipt_name = '.cafin_archive.json'
    receipt = destination / receipt_name
    restart = 'Start a fresh runtime and rerun setup with the intended code ZIP.'
    if destination.exists():
        if not receipt.is_file():
            raise ValueError('Existing project has no completed extraction receipt. ' + restart)
        recorded = json.loads(receipt.read_text(encoding='utf-8'))
        if recorded['archive_sha256'] != archive_hash:
            raise ValueError('Code ZIP changed since extraction. ' + restart)
        for name, expected in recorded['files'].items():
            target = (destination / name).resolve()
            if not target.is_relative_to(destination.resolve()):
                raise ValueError('Unsafe receipt path')
            if not target.is_file() or file_sha256(target) != expected:
                raise ValueError('Extracted project file changed: ' + name + '. ' + restart)
        return recorded

    destination.parent.mkdir(parents=True, exist_ok=True)
    # Stage on the same filesystem; a failed extraction never publishes PROJECT.
    with tempfile.TemporaryDirectory(prefix='cafin_extract_', dir=destination.parent) as temporary:
        stage = Path(temporary) / 'project'
        stage.mkdir()
        with zipfile.ZipFile(archive_path) as archive:
            names = archive.namelist()
            if len(names) != len(set(names)) or receipt_name in names:
                raise ValueError('Duplicate or reserved archive member')
            for member in archive.infolist():
                if not (stage / member.filename).resolve().is_relative_to(stage.resolve()):
                    raise ValueError('Unsafe archive path')
            for required in ('pyproject.toml', 'src/cafin/__init__.py', 'src/cafin/cli.py'):
                if required not in names:
                    raise ValueError('Incomplete code ZIP: missing ' + required)
            archive.extractall(stage)
            files = {member.filename: file_sha256(stage / member.filename)
                     for member in archive.infolist() if not member.is_dir()}
        recorded = {'archive_sha256': archive_hash, 'files': files}
        (stage / receipt_name).write_text(json.dumps(recorded, indent=2), encoding='utf-8')
        stage.rename(destination)
    return recorded


DRIVE_ROOT = Path('/content/drive/MyDrive/CAFIN')
PROJECT_ZIP = DRIVE_ROOT / 'cafin_ipinyou_code.zip'
PROJECT = Path('/content/cafin_project')
source_receipt = restore_project(PROJECT_ZIP, PROJECT)
print('Code ZIP SHA-256:', source_receipt['archive_sha256'])
os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)
if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi'], check=False)
else:
    print('nvidia-smi unavailable; CPU checks can still run.')

def run_cli(*args):
    subprocess.run([sys.executable, '-m', 'cafin', *map(str, args)], check=True)

def save_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2), encoding='utf-8')

In [ ]:
import shutil
DATA_DIR = Path('/content/cafin_data/standardized')
CACHE = DRIVE_ROOT / 'cache' / 'standardized.zip'
if not (DATA_DIR / 'manifest.json').exists() and CACHE.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(CACHE) as archive:
        for member in archive.infolist():
            if not (DATA_DIR / member.filename).resolve().is_relative_to(DATA_DIR.resolve()):
                raise ValueError('Unsafe cache path')
        archive.extractall(DATA_DIR)
print('Prepared data:', (DATA_DIR / 'manifest.json').exists())

In [ ]:
MODEL, EXPERIMENT, SEED = 'CAFIN', 'v1', 11
RUN_DIR = DRIVE_ROOT / 'runs' / f'{MODEL}_{EXPERIMENT}_seed{SEED}'
# For a sweep, set RUN_DIR to its exact experiments/.../runs/<run_id> folder.
manifest = json.loads((DATA_DIR / 'manifest.json').read_text())
POLICY = RUN_DIR / 'bidder_policy.json'
if manifest['protocol'] == 'chronological' and not POLICY.exists():
    run_cli('evaluate', '--run', RUN_DIR, '--split', 'valid', '--data-dir', DATA_DIR)
    run_cli('tune-bidder', '--run', RUN_DIR, '--config', PROJECT/'configs/rtb.json', '--output', POLICY)

In [ ]:
RUN_FINAL_TEST = False
if not RUN_FINAL_TEST:
    print('Test remains locked. Finish model/bidder selection before changing RUN_FINAL_TEST.')
else:
    run_cli('freeze', '--run', RUN_DIR)
    run_cli('evaluate', '--run', RUN_DIR, '--split', 'test', '--allow-test', '--data-dir', DATA_DIR)
    run_cli('group-metrics', '--run', RUN_DIR, '--split', 'test', '--output', RUN_DIR/'campaign_season_metrics.json')
    if manifest['protocol'] == 'chronological':
        run_cli('replay', '--run', RUN_DIR, '--policy', POLICY, '--output', RUN_DIR/'rtb_results.json')
    run_cli('summarize', '--runs', RUN_DIR.parent, '--split', 'test', '--output', RUN_DIR.parent.parent/'results/test_summary.csv')

Bảng tổng hợp phân tách dataset hash, code và cấu hình; không trộn các seed có hyperparameter khác nhau. Standard deviation dùng mẫu (`ddof=1`); một seed không có std.
RTB là replay trên historical observed impressions; eCPC giữ đơn vị giá gốc của dataset, không tự chuyển thành tiền tệ.
Không có click → eCPC null. Campaign thiếu cả hai lớp → ROC-AUC null.